In [36]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    temperature=0,
    groq_api_key='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',
    model_name="llama-3.3-70b-versatile",
)
response=llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [111]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://amazon.jobs/content/en/job-categories/software-development")
page_data = loader.load().pop().page_content
print(page_data)

























Software Development
Amazon JobsTeamsLocationsJob categoriesResourcesAccommodationsBenefitsInclusive experiencesInterviewing at AmazonLeadership PrinciplesWorking at AmazonFAQLocaleLanguage: en-USAmazon JobsSearchSearchLocaleLanguage: en-US Search for jobsTitle or keywordLocationSearchSearch Software DevelopmentTechnology is in everything we do. We rely on it to provide the excellence our customers deserve. And that’s why our software engineers are crucial to every aspect of our operations.Building impactful solutions at large scaleDo you want to use your knowledge to build software solutions? Do you enjoy solving complex problems? In a fast-paced and innovative environment, there are so many projects you can work on. And you can take pride in knowing you’ll improve the lives of millions of people around the world.With us, you’ll enhance your skills and work with other talented minds. You’ll have freedom to explore and turn your ideas into reality. The learning 

In [112]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [113]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)

# Normalize: if it's a list, take the first element
if isinstance(json_res, list):
    json_res = json_res[0]

print(type(json_res))   # should now be <class 'dict'>
print(json_res)


<class 'dict'>
{'role': 'Software Development Engineer', 'experience': 'II', 'skills': 'Technical skills, soft skills', 'description': 'Build impactful solutions at large scale, solve complex problems, improve lives of millions of people around the world'}


In [114]:
type(json_res)

dict

In [115]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [116]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [117]:
links = collection.query(query_texts=['Computer Science',], n_results=3).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}]]

In [118]:
job

{'role': 'Network Fabric Test Engineer',
 'experience': 'Early',
 'skills': ['Computer Science',
  'Electrical Engineering',
  'Networking protocols',
  'Test methodologies'],
 'description': 'Completing work as directed, and collaborating with teammates; developing knowledge of relevant concepts and processes.'}

In [119]:
job = json_res
job['skills']

'Technical skills, soft skills'

In [120]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}
    
    ### INSTRUCTION:
    You are Aditi, a recent graduate in Computer Science. 
    Write a professional job application email ONLY for roles related to Computer Science, 
    Software Engineering, or Programming. 
    Highlight your enthusiasm to learn, your academic background, and your skills in Python, C++, and pandas. 
    Keep the tone polite, confident, and concise. 
    Do not add any preamble.
    
    ### EMAIL (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Application for Software Development Engineer II Role

Dear Hiring Manager,

I am excited to apply for the Software Development Engineer II position at your esteemed organization. With a strong academic background in Computer Science and a passion for building innovative solutions, I am confident that I would be an excellent fit for this role.

Throughout my undergraduate studies, I developed a solid foundation in programming principles, data structures, and software engineering. I am proficient in languages such as Python and C++, and have experience working with popular libraries like pandas. I am eager to apply my skills and knowledge to real-world problems and contribute to the development of impactful solutions that can improve the lives of millions of people worldwide.

As a recent graduate, I am enthusiastic about learning and growing with a dynamic team. I am impressed by the company's commitment to building large-scale solutions and solving complex problems. I am exci